# Fine-tune ViDeBERTa-xsmall cho chuẩn hóa địa chỉ

Chọn **Runtime → Change runtime type → GPU**, rồi chạy các ô theo thứ tự. Upload `nlp_dia_chi_colab_v1.zip` ở ô 2. Không cần GitHub hoặc Hugging Face token.

**Mặc định:** 5 epoch trên 4.841 mẫu, 1.169 mẫu riêng cho ranker; dev gốc 1.136 và dev khó 464 câu. Không có tập test cuối trong gói. Dự trù train GPU 10–30 phút, ETA được in sau 10 batch; đây là ước lượng, chưa đo trên GPU Colab. Dùng Drive để giữ checkpoint khi runtime bị ngắt.

## 1. Kiểm tra GPU

In [ ]:
import sys, subprocess, torch
assert torch.cuda.is_available(), "Bật GPU ở Runtime → Change runtime type trước khi chạy tiếp"
print("Torch:", torch.__version__, "GPU:", torch.cuda.get_device_name())
subprocess.run(["nvidia-smi"], check=True)


Torch: 2.11.0+cu130 GPU: Tesla T4


CompletedProcess(args=['nvidia-smi'], returncode=0)

## 2. Upload và giải nén gói dự án
Chỉ chọn ZIP đi kèm notebook này. Nếu chạy lại trong cùng runtime, thư mục đã giải nén sẽ được dùng lại và kiểm checksum ở ô 4.

In [7]:
from pathlib import Path
import zipfile, os, sys

with zipfile.ZipFile("/content/nlp_dia_chi_colab_v1.zip") as z:
    z.extractall("/content")

PROJECT = Path("/content/nlp_dia_chi_colab")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print(PROJECT)


FileNotFoundError: [Errno 2] No such file or directory: '/content/nlp_dia_chi_colab_v1.zip'

## 3. Cài thư viện
Giữ nguyên PyTorch CUDA của Colab. Không chạy `requirements-neural.txt` vì đó là môi trường CPU cũ.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "colab/requirements.txt"], check=True)
assert torch.cuda.is_available(), "CUDA phải còn hoạt động sau cài thư viện"


## 4. Kiểm tra gói và smoke test
Các test dùng backbone tí hon để kiểm tra gradient, lưu/nạp, Unicode và pipeline; chưa phải fine-tune đầy đủ.

In [ ]:
subprocess.run([sys.executable, "-m", "colab.verify_bundle", "--root", str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "unittest", "tests.test_videberta_colab", "tests.test_char", "tests.test_pipeline", "tests.test_pipeline_v3", "tests.test_crf", "tests.test_danh_gia"], check=True)


## 5. Nơi lưu và cấu hình
Drive được khuyên dùng để giữ checkpoint. Nếu không dùng Drive, cần tải kết quả về trước khi Colab kết thúc. Giữ nguyên cấu hình khi resume. Muốn đổi batch/epoch thì đổi RUN_ID để tạo một lần thử mới. Nếu hết VRAM, dùng BATCH_SIZE=4 và GRAD_ACC=4 trong một run mới.

In [ ]:
USE_DRIVE = True
RUN_ID = "videberta_v1_01"
EPOCHS = 5
BATCH_SIZE = 8
GRAD_ACC = 2
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/nlp_chuan_hoa_dia_chi") / RUN_ID
else:
    RUN_DIR = Path("/content/runs") / RUN_ID
print("Run:", RUN_DIR, "Resume:", RUN_DIR.exists())


## 6. Fine-tune GPU
Checkpoint pretrained được tải trực tiếp từ commit đã chốt. Lưu model và resume sau mỗi epoch. Resume quay lại epoch hoàn chỉnh gần nhất; nếu chưa xong epoch nào thì bắt đầu lại epoch 1. Không dùng dev để chọn checkpoint.

In [ ]:
cmd = [sys.executable, "-u", "-m", "colab.train_videberta", "--data-root", "bundle_data", "--run-dir", str(RUN_DIR), "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--gradient-accumulation", str(GRAD_ACC)]
if RUN_DIR.exists():
    cmd.append("--resume")
subprocess.run(cmd, check=True)
print((RUN_DIR / "training.json").read_text())


## 7. Train ranker và đánh giá CPU
Ranker học trên phần train riêng; đo dev với luật v3, CRF cũ, ViDeBERTa dùng điểm tay và dùng ranker. **Độ trễ ở đây là CPU Colab**, không phải CPU laptop. Tỷ lệ khớp là với nhãn máy/tổng hợp, chưa phải gold. Muốn đo lại, đổi tên thư mục báo cáo để giữ lịch sử.

In [ ]:
REPORT_DIR = RUN_DIR / "reports_cpu_colab_01"
subprocess.run([sys.executable, "-u", "-m", "colab.evaluate_videberta", "--run-dir", str(RUN_DIR), "--output", str(REPORT_DIR)], check=True)
print((REPORT_DIR / "BAO_CAO.md").read_text())


## 8. Thử một địa chỉ trên CPU

In [ ]:
subprocess.run([sys.executable, "-m", "src", "Số 12, Đường Lê Lợi, Phường Trúc Bạch, Quận Ba Đình, Hà Nội", "--videberta-model", str(RUN_DIR / "model"), "--ranker", str(RUN_DIR / "ranker.txt"), "--threads", "1"], check=True)


## 9. Xuất và tải kết quả
ZIP chứa model, tokenizer, ranker, dữ liệu snapshot, cấu hình và báo cáo. Optimizer/resume lớn được giữ trên Drive, không đưa vào ZIP tải về. Giải nén ZIP kết quả vào `D:\nlp_chuan_hoa_dia_chi\mo_hinh\thu_nghiem\`; xem lệnh chạy CPU trong hướng dẫn đi kèm gói ban đầu.

In [ ]:
from datetime import datetime
RESULT_ZIP = Path("/content") / (RUN_ID + "_ket_qua_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".zip")
subprocess.run([sys.executable, "-m", "colab.export_results", "--run-dir", str(RUN_DIR), "--output", str(RESULT_ZIP)], check=True)
files.download(str(RESULT_ZIP))
